# Study 5

Compared point and range delivery estimates and tested whether perceived unexpected lateness helped explain the difference in satisfaction.

In [1]:
# Import packages and locate the data folder.
from pathlib import Path
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import binomtest, t
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/Study 1.xlsx').exists())
os.environ['MPLCONFIGDIR'] = str(ROOT / '.cache/matplotlib')
ITEMS = ['sb_like', 'sb_info', 'sb_useful', 'sb_accurate', 'sb_trust']
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

In [2]:
# Load data, apply exclusions, and calculate satisfaction.
d = pd.read_excel(ROOT / 'data/raw/Study 5.xlsx')
raw_n = len(d)

d = d.loc[(d.ExcludeDuplicateIP != 1) & (d.ExcludeDuplicateID != 1)].copy()
assert len(d) == 468
d['satisfaction'] = d[ITEMS].mean(axis=1)
rows, descriptions = [], []
display(pd.DataFrame([{'raw_n':raw_n, 'replicated_n':len(d), 'paper_n':468, 'match':len(d)==468}]))

,raw_n,replicated_n,paper_n,match
0,498,468,468,True


In [3]:
# Define helpers for regressions and paper comparisons.
def record(study, label, fit, term, target_b=None, target_se=None, sign=1):
    ci = np.sort(np.asarray(fit.conf_int().loc[term]) * sign)
    b, se = (sign * fit.params[term], fit.bse[term])
    row = {'study': study, 'analysis': label, 'n': int(fit.nobs), 'b': b, 'se': se, 'p': fit.pvalues[term], 'ci_low': ci[0], 'ci_high': ci[1], 'published_b': target_b, 'published_se': target_se, 'b_matches_2dp': None if target_b is None else abs(b - target_b) <= 0.0050001, 'se_matches_2dp': None if target_se is None else abs(se - target_se) <= 0.0050001}
    rows.append(row)

def ols(d, y, terms):
    return sm.OLS(d[y], sm.add_constant(d[terms].astype(float)), missing='drop').fit()

In [4]:
# Estimate the total range-versus-point effect on satisfaction.
d['range'] = d.Condition.map({1:-.5, 2:.5})
record(5, 'range effect on satisfaction', ols(d,'satisfaction',['range']), 'range', 0.82, .11)

In [5]:
# Estimate the mediation paths and bootstrap the indirect effect.
record(5,'range effect on lateness',ols(d,'late',['range']),'range',-1.67,.11)
record(5,'unadjusted lateness association',ols(d,'satisfaction',['late']),'late',-.44,.04)
record(5,'direct range effect',ols(d,'satisfaction',['range','late']),'range',.13,.12)
complete = d[['satisfaction','range','late']].dropna().to_numpy(dtype=float)
def indirect(a):
    y, treatment, mediator = a.T
    total = np.linalg.lstsq(np.column_stack([np.ones(len(a)), treatment]),y,rcond=None)[0][1]
    direct = np.linalg.lstsq(np.column_stack([np.ones(len(a)), treatment, mediator]),y,rcond=None)[0][1]
    return total-direct
rng = np.random.default_rng(20261027)
boot = np.array([indirect(complete[rng.integers(0,len(complete),len(complete))]) for _ in range(1000)])
point = indirect(complete)
ci = point + np.array([-1,1])*1.95996398454*boot.std(ddof=1)
# Original bootstrap was unseeded; exact resamples differ. Normal interval matches Stata's default.
display(pd.DataFrame([{'indirect':point,'paper_indirect':.69,'ci_low':ci[0],'paper_ci_low':.51,'ci_high':ci[1],'paper_ci_high':.87,'replicates':1000,'seed':20261027}]))
assert abs(point-.69)<=.0050001

,indirect,paper_indirect,ci_low,paper_ci_low,ci_high,paper_ci_high,replicates,seed
0,0.6893,0.69,0.5108,0.51,0.8679,0.87,1000,20261027


In [6]:
# Display estimates beside the paper and check agreement at reported precision.
results = pd.DataFrame(rows)
checked = results.published_b.notna()
assert results.loc[checked, 'b_matches_2dp'].all()
assert results.loc[checked, 'se_matches_2dp'].all()
display(results[['analysis','n','b','published_b','se','published_se','p','b_matches_2dp','se_matches_2dp']])

,analysis,n,b,published_b,se,published_se,p,b_matches_2dp,se_matches_2dp
0,range effect on satisfaction,468,0.8218,0.82,0.1089,0.11,2.3512e-13,True,True
1,range effect on lateness,468,-1.6735,-1.67,0.1055,0.11,1.2650e-45,True,True
2,unadjusted lateness association,468,-0.4397,-0.44,0.0354,0.04,7.7055e-31,True,True
3,direct range effect,468,0.1325,0.13,0.1240,0.12,2.8608e-01,True,True


In [7]:
# Display sample sizes, means, and standard deviations by condition.
display(d.groupby('Condition')[['satisfaction']+['late']].agg(['count','mean','std']))

satisfaction                  late                
                 count    mean     std count    mean     std
Condition                                                   
1                  233  4.3382  1.2834   233  4.1416  1.0953
2                  235  5.1600  1.0626   235  2.4681  1.1849